# 06 — AdaBoost

**Project:** Healthcare Readmission Intelligence  
**Input:** `data/features/train_features.csv` and `data/features/test_features.csv`  
**Output:** Trained AdaBoost model + evaluation report + comparison with baselines

## Objective

AdaBoost (Adaptive Boosting) is our first ensemble method beyond bagging. Unlike Random Forest 
which trains trees independently, AdaBoost trains weak learners **sequentially** — each new tree 
focuses on the samples the previous one got wrong.

We will:

| Step | What |
| ---- | ---- |
| 1 | Train a default AdaBoost with Decision Tree stumps |
| 2 | Tune `n_estimators`, `learning_rate`, and base estimator depth |
| 3 | Evaluate against baseline models using the same metrics |
| 4 | Analyze feature importances |

> AdaBoost with `algorithm='SAMME'` supports `class_weight` on the base estimator — 
> this is critical for our 88.7/11.3 imbalanced dataset.


In [1]:
import warnings
import os
import sys
import yaml

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    f1_score,
    recall_score,
    precision_score,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
)

warnings.filterwarnings("ignore")
plt.style.use("ggplot")
pd.set_option("display.max_columns", None)


# 1. Load the dataset

In [2]:
sys.path.append(os.path.abspath('..'))
from scripts.evaluate import evaluate_classifier

CONFIG_PATH = '../configs/paths.yaml'

with open(CONFIG_PATH, 'r') as file:
    paths = yaml.safe_load(file)

df_train = pd.read_csv(paths['features_data']['train_data'])
df_test  = pd.read_csv(paths['features_data']['test_data'])

print(f"Train shape: {df_train.shape}")
print(f"Test shape : {df_test.shape}")
df_train.head()

Train shape: (79219, 38)
Test shape : (20124, 38)


,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,number_diagnoses,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,glipizide,glyburide,pioglitazone,rosiglitazone,acarbose,insulin,glyburide-metformin,glipizide-metformin,change,diabetesMed,diag_1_group,diag_2_group,diag_3_group,medical_specialty_group,total_prior_visits,high_utilizer,A1C_clinical_response,readmitted_binary
0,Caucasian,Female,0,Other_Type,Other_Discharge,Referral,1,Unknown,41,0,1,0,0,0,1,No,No,No,No,No,No,No,No,No,No,0,No,No,No,No,Diabetes,Unknown,Unknown,Pediatrics,0,0,Not_Measured,0
1,Caucasian,Female,1,Emergency,Discharged_Home,Emergency_Room,3,Unknown,59,0,18,0,0,0,9,No,No,No,No,No,No,No,No,No,No,3,No,No,Yes,Yes,Other,Diabetes,Other,Missing,0,0,Not_Measured,0
2,AfricanAmerican,Female,2,Emergency,Discharged_Home,Emergency_Room,2,Unknown,11,5,13,2,0,1,6,No,No,No,No,No,Yes,No,No,No,No,0,No,No,No,Yes,Other,Diabetes,Other,Missing,3,1,Not_Measured,0
3,Caucasian,Male,3,Emergency,Discharged_Home,Emergency_Room,2,Unknown,44,1,16,0,0,0,7,No,No,No,No,No,No,No,No,No,No,3,No,No,Yes,Yes,Other,Diabetes,Circulatory,Missing,0,0,Not_Measured,0
4,Caucasian,Male,5,Urgent,Discharged_Home,Referral,3,Unknown,31,6,16,0,0,0,9,No,No,No,No,No,No,No,No,No,No,1,No,No,No,Yes,Circulatory,Circulatory,Diabetes,Missing,0,0,Not_Measured,0


In [3]:
X_train = df_train.drop(columns=['readmitted_binary'])
X_test  = df_test.drop(columns=['readmitted_binary'])

y_train = df_train['readmitted_binary']
y_test  = df_test['readmitted_binary']

print(f"X_train: {X_train.shape}  |  y_train: {y_train.shape}")
print(f"X_test : {X_test.shape}   |  y_test : {y_test.shape}")

X_train: (79219, 37)  |  y_train: (79219,)
X_test : (20124, 37)   |  y_test : (20124,)


In [4]:
from scripts.preprocessor import get_preprocessor

preprocessor = get_preprocessor()

In [9]:
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('adaboost',AdaBoostClassifier(
        estimator=DecisionTreeClassifier(
            max_depth = 1,
            class_weight='balanced',
            random_state = 30
        )
    ))
])

default_result = evaluate_classifier(
    'AdaBoost (default)',
    pipeline,
    X_train, X_test,
    y_train, y_test
)

print("AdaBoost — Default Parameters")
print(classification_report(y_test, default_result['_test_pred']))

AdaBoost — Default Parameters
              precision    recall  f1-score   support

           0       0.89      0.88      0.89     17783
           1       0.16      0.16      0.16      2341

    accuracy                           0.80     20124
   macro avg       0.52      0.52      0.52     20124
weighted avg       0.80      0.80      0.80     20124



In [11]:
param_grid = {
    'adaboost__n_estimators': [50, 100, 200, 300],
    'adaboost__learning_rate': [0.01, 0.1, 0.5, 1.0],
    'adaboost__estimator__max_depth': [1, 2, 3],
}

grid = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring='f1',
    cv=5,
    verbose=3
)

print("Running grid search...")
grid.fit(X_train, y_train)

Running coarse grid search...
Fitting 5 folds for each of 48 candidates, totalling 240 fits
[CV 1/5] END adaboost__estimator__max_depth=1, adaboost__learning_rate=0.01, adaboost__n_estimators=50;, score=0.228 total time=  19.7s
[CV 2/5] END adaboost__estimator__max_depth=1, adaboost__learning_rate=0.01, adaboost__n_estimators=50;, score=0.263 total time=  20.9s
[CV 3/5] END adaboost__estimator__max_depth=1, adaboost__learning_rate=0.01, adaboost__n_estimators=50;, score=0.259 total time=  23.9s
[CV 4/5] END adaboost__estimator__max_depth=1, adaboost__learning_rate=0.01, adaboost__n_estimators=50;, score=0.268 total time=  20.0s
[CV 5/5] END adaboost__estimator__max_depth=1, adaboost__learning_rate=0.01, adaboost__n_estimators=50;, score=0.246 total time=  21.7s
[CV 1/5] END adaboost__estimator__max_depth=1, adaboost__learning_rate=0.01, adaboost__n_estimators=100;, score=0.228 total time=  18.3s
[CV 2/5] END adaboost__estimator__max_depth=1, adaboost__learning_rate=0.01, adaboost__n_es

In [13]:
print("\nBest Parameters :", grid.best_params_)
print("Best CV F1 Score :", round(grid.best_score_, 4))


Best Parameters : {'adaboost__estimator__max_depth': 3, 'adaboost__learning_rate': 0.5, 'adaboost__n_estimators': 50}
Best CV F1 Score : 0.2678
